# Odor Identity Interpretability — Closing the Last GLM Gap

**This notebook closes the one remaining gap for the GLM to be fully complete.** InSeq/OutSeq
interpretability was done in notebook 015; this does the equivalent for Odor Identity, at each rat's own
winning window from notebook 016 (a single continuous window of 1000-1750ms starting at Poke-In, NOT the
short 250ms InSeq/OutSeq window). After this notebook, both tasks have final models AND interpretability
for all 5 rats, the GLM track itself is complete.

**One expected difference from notebook 015 going in:** InSeq/OutSeq's 250ms window was too short to
reliably resolve delta-band (1-4 Hz) frequencies, producing unreliable, sometimes exactly-zero
importance values for that band. Odor Identity's windows are much longer (1000-1750ms), comfortably long
enough to resolve delta properly. This notebook checks directly whether that resolution problem actually
disappears here, rather than assuming it does.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from src.preprocessing import build_labels, get_sampling_rate


## 1. Each Rat's Winning Window (From Notebook 016)

In [ ]:
WINNING_WINDOWS = {
    '080718_mitt':       {'length_ms': 1750},
    '081106_barat':      {'length_ms': 1000},
    '090212_stella':     {'length_ms': 1750},
    '090212_superchris': {'length_ms': 1500},
    '090420_buchanan':   {'length_ms': 1500},
}

BANDS = {
    'delta': (1, 4),
    'theta': (4, 12),
    'beta': (12, 30),
    'low_gamma': (30, 80),
    'high_gamma': (80, 150),
}
BAND_NAMES = list(BANDS.keys())

raw_dir = Path('../data/raw')
session_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])


## 2. Helper Functions (Unchanged From Prior Notebooks)

In [ ]:
def extract_window_at_offset(lfp_data, timebin, poke_idx, offset_ms, window_ms, target_samples):
    poke_time = timebin[poke_idx]
    start_time = poke_time + offset_ms / 1000
    end_time = start_time + window_ms / 1000
    start_idx = np.searchsorted(timebin, start_time)
    end_idx = np.searchsorted(timebin, end_time)
    if end_idx - start_idx < 2:
        return None
    raw_window = lfp_data[:, start_idx:end_idx]
    n_channels, n_raw = raw_window.shape
    old_x = np.linspace(0, 1, n_raw)
    new_x = np.linspace(0, 1, target_samples)
    resampled = np.zeros((n_channels, target_samples))
    for ch in range(n_channels):
        resampled[ch] = np.interp(new_x, old_x, raw_window[ch])
    return resampled


def band_power_features_named(windows, fs, bands, channel_names):
    n_trials, n_channels, n_samples = windows.shape
    freqs = np.fft.rfftfreq(n_samples, d=1 / fs)
    band_masks = {name: (freqs >= lo) & (freqs < hi) for name, (lo, hi) in bands.items()}
    n_features = n_channels * len(bands)
    X = np.zeros((n_trials, n_features))
    feature_names = []
    col = 0
    for ch in range(n_channels):
        fft_vals = np.fft.rfft(windows[:, ch, :], axis=1)
        power = np.abs(fft_vals) ** 2
        for band_name, mask in band_masks.items():
            X[:, col] = power[:, mask].sum(axis=1)
            feature_names.append(f"{channel_names[ch]}_{band_name}")
            col += 1
    return X, feature_names


## 3. Train Each Rat's Final Odor Identity Model and Extract Coefficients

Refit on all InSeq trials (not cross-validated here, purely to inspect what the model learned). For a
5-class problem, `coef_` has one row per class; we average |coefficient| across all 5 classes to get one
overall importance score per feature, same approach used for odor interpretability back in notebook 05.


In [ ]:
per_rat_importance = {}

for session_dir in session_dirs:
    session_name = session_dir.name
    length_ms = WINNING_WINDOWS[session_name]['length_ms']

    bvr = np.load(session_dir / f'{session_name}_bvr.npz', allow_pickle=True)
    bvr_data = bvr['data']
    bvr_keys = bvr['keys'].tolist()
    lfp = np.load(session_dir / f'{session_name}_lfp.npz', allow_pickle=True)
    lfp_data = lfp['data']
    lfp_keys = lfp['keys'].tolist()

    timebin = bvr_data[bvr_keys.index('TimeBin')]
    avg_fs = get_sampling_rate(bvr_data, bvr_keys)
    labels = build_labels(bvr_data, bvr_keys)
    target_samples = int(round(length_ms / 1000 * avg_fs))

    windows, kept_idx = [], []
    for t in labels['trial_idx']:
        w = extract_window_at_offset(lfp_data, timebin, t, 0, length_ms, target_samples)
        if w is not None:
            windows.append(w)
            kept_idx.append(t)
    windows = np.stack(windows, axis=0)
    kept_mask = np.isin(labels['trial_idx'], np.array(kept_idx))
    inseq_outseq = labels['inseq_outseq'][kept_mask]
    odor_id = labels['odor_id'][kept_mask]

    inseq_mask = (inseq_outseq == 1)
    X, feature_names = band_power_features_named(windows[inseq_mask], avg_fs, BANDS, lfp_keys)
    X_log = np.log1p(X)
    y = odor_id[inseq_mask]

    pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
    pipe.fit(X_log, y)
    coefs = pipe.named_steps['clf'].coef_  # shape (5 classes, n_features)
    importance = np.abs(coefs).mean(axis=0)

    per_rat_importance[session_name] = {
        'feature_names': feature_names,
        'importance': importance,
        'coefs': coefs,
        'n_channels': windows.shape[1],
        'window_length_ms': length_ms,
    }
    print(f"{session_name:20s} fit complete, window={length_ms}ms, {windows.shape[1]} channels")


## 4. Top Features Per Rat


In [ ]:
rats = list(per_rat_importance.keys())
fig, axes = plt.subplots(1, 5, figsize=(24, 6))

for ax, session_name in zip(axes, rats):
    r = per_rat_importance[session_name]
    order = np.argsort(r['importance'])[::-1][:10]
    ax.barh(range(len(order)), r['importance'][order], color='#4C72B0')
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([r['feature_names'][i] for i in order], fontsize=8)
    ax.invert_yaxis()
    ax.set_title(f"{session_name.split('_')[1]} ({r['window_length_ms']}ms)", fontsize=10)

plt.suptitle('Top 10 features per rat, Odor Identity, at each rat\'s winning window')
plt.tight_layout()
plt.show()


## 5. Cross-Rat Band Importance Comparison, Odor Identity

The direct comparison against notebook 015's InSeq/OutSeq version: same chart type, different task and
windows. Also checks whether the delta-band resolution problem from InSeq/OutSeq's short window actually
disappears here, as expected, given these windows are 4-7x longer.


In [ ]:
band_importance_matrix = np.zeros((len(rats), len(BAND_NAMES)))

for row, session_name in enumerate(rats):
    r = per_rat_importance[session_name]
    for col, band_name in enumerate(BAND_NAMES):
        band_vals = [r['importance'][i] for i, name in enumerate(r['feature_names']) if name.endswith(f'_{band_name}')]
        band_importance_matrix[row, col] = np.mean(band_vals)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(band_importance_matrix, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(BAND_NAMES))); ax.set_xticklabels(BAND_NAMES)
ax.set_yticks(range(len(rats))); ax.set_yticklabels([s.split('_')[1] for s in rats])
ax.set_title('Mean |coefficient| by band and rat (Odor Identity)')
plt.colorbar(im, ax=ax, label='mean |coefficient|')
for i in range(len(rats)):
    for j in range(len(BAND_NAMES)):
        ax.text(j, i, f'{band_importance_matrix[i, j]:.2f}', ha='center', va='center',
                 color='white' if band_importance_matrix[i, j] < band_importance_matrix.max()/2 else 'black', fontsize=9)
plt.tight_layout()
plt.show()

overall_ranking = band_importance_matrix.mean(axis=0)
ranking_order = np.argsort(overall_ranking)[::-1]
print("Band importance, averaged across all 5 rats, ranked highest to lowest:")
for i in ranking_order:
    print(f"  {BAND_NAMES[i]:12s} {overall_ranking[i]:.4f}")

n_zero_delta = sum(1 for row in range(len(rats)) if band_importance_matrix[row, BAND_NAMES.index('delta')] == 0.0)
print(f"\nRats with exactly zero delta importance: {n_zero_delta} of {len(rats)} "
      f"(compare to 2 of 5 in notebook 015's short-window InSeq/OutSeq analysis)")


## 6. Compare Against InSeq/OutSeq's Band Pattern (Notebook 015)

Are the same bands important for both tasks, or does each task rely on different frequency information?


In [ ]:
inseq_band_importance = np.array([
    [0.3965, 0.3872, 0.2225, 0.3741, 0.5020],  # Mitt
    [0.0000, 0.3178, 0.2674, 0.3784, 0.3573],  # Barat
    [0.3563, 0.3844, 0.3460, 0.2074, 0.3599],  # Stella
    [0.0000, 0.4012, 0.4585, 0.3758, 0.4793],  # Superchris
    [0.3060, 0.4992, 0.4616, 0.4202, 0.6262],  # Buchanan
])
inseq_overall = inseq_band_importance.mean(axis=0)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(BAND_NAMES))
width = 0.35
ax.bar(x - width/2, inseq_overall, width, label='InSeq/OutSeq (notebook 015)', color='#4C72B0')
ax.bar(x + width/2, overall_ranking, width, label='Odor Identity (this notebook)', color='#DD8452')
ax.set_xticks(x)
ax.set_xticklabels(BAND_NAMES)
ax.set_ylabel('Mean |coefficient|, averaged across all 5 rats')
ax.set_title('Band importance: InSeq/OutSeq vs. Odor Identity')
ax.legend()
plt.tight_layout()
plt.show()


## 7. Text-Only Results Export


In [ ]:
import json as _json
import os

results_summary = {
    "purpose": "Odor Identity interpretability, closing the last GLM gap",
    "bands": {name: list(rng) for name, rng in BANDS.items()},
    "windows_used": {name: WINNING_WINDOWS[name]['length_ms'] for name in rats},
    "band_importance_by_rat": {
        rats[row]: {BAND_NAMES[col]: round(float(band_importance_matrix[row, col]), 4) for col in range(len(BAND_NAMES))}
        for row in range(len(rats))
    },
    "overall_band_ranking": [
        {"band": BAND_NAMES[i], "mean_importance": round(float(overall_ranking[i]), 4)}
        for i in ranking_order
    ],
    "n_rats_zero_delta": int(n_zero_delta),
    "top_10_features_by_rat": {
        session_name: [
            {"feature": r['feature_names'][i], "importance": round(float(r['importance'][i]), 4)}
            for i in np.argsort(r['importance'])[::-1][:10]
        ]
        for session_name, r in per_rat_importance.items()
    },
}

print(_json.dumps(results_summary, indent=2))

os.makedirs('../outputs/logs', exist_ok=True)
with open('../outputs/logs/notebook018_odor_interpretability_results.json', 'w') as f:
    _json.dump(results_summary, f, indent=2)
print("\nSaved to outputs/logs/notebook018_odor_interpretability_results.json")


## 8. Written Summary Report

**Objective**

Close the last remaining GLM gap: produce an interpretability analysis for Odor Identity, matching the
standard already set for InSeq/OutSeq in notebook 015, at each rat's actual winning window from
notebook 016 (a longer, 1000-1750ms window starting at Poke-In, not the short 250ms InSeq/OutSeq
window).

**Method**

Each rat's final Odor Identity model (logistic regression, 5-class) was refit on all InSeq trials at
its winning window length. For the 5-class problem, coefficient magnitude was averaged across all 5
odor classes to get one importance score per channel/band feature, then averaged across channels within
each band to compare band-level importance, directly comparable to notebook 015's InSeq/OutSeq analysis.

**Results**

*(Fill in after running Sections 5-6.)* Which band ranks highest for Odor Identity? Does the delta-band
resolution problem from InSeq/OutSeq's short window actually disappear here, given the much longer
windows? Are the same bands important for both tasks, or does each rely on different frequency
information?

**Interpretation**

*(Fill in after review.)* A shared important band across both tasks (e.g. if high gamma or theta ranks
highly for both) would suggest a general marker of task-relevant hippocampal processing, not specific to
either judgment. A different pattern per task would suggest InSeq/OutSeq and Odor Identity rely on
distinguishable neural signatures, consistent with the fact that they already need different time
windows to decode well.

**Status: GLM Track Complete**

With this notebook, both InSeq/OutSeq and Odor Identity now have final, validated, statistically-tested
models AND interpretability analyses, for all 5 rats. This is everything the GLM approach can offer for
this project as currently scoped. RNN retraining, multi-rat pooling, GNN, and cross-rat statistical
comparison remain open, but these are extensions beyond the GLM, not gaps within it.
